# 📘 学习注释版：Silver CRM Product

**Input：** `workspace.bronze.crm_prd_info`  
**Output：** `workspace.silver.crm_products`

主要学习：
- Product Key 解析
- NULL Cost → 0
- Product Line Code 标准化
- 日期类型转换
- 字段统一命名


#Initialization

## 🧰 学习说明：导入 PySpark 函数/类型

这里仅准备后续清洗需要的 API，例如：
- `col()`：引用列
- `trim()`：去前后空格
- `StringType`：判断字符串类型
- `F.when()`：类似 SQL `CASE WHEN`

这一 Cell 不改变数据。


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Read Bronze table

## 📖 学习说明：读取 Bronze Table

**Input：** `workspace.bronze.crm_prd_info`  
**Process：** `spark.table()` 把 Catalog 中的 Table 读取成 DataFrame  
**Output：** 变量 `df`

后续所有 Silver 清洗都在这个 DataFrame 上进行。


In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

#Silver Transformations

##Trimming

## ✂️ 学习说明：批量 Trim 字符串

遍历 DataFrame 所有字段：

```text
如果字段类型 = String
→ trim()
→ 去掉前后空格
```

真实数据中 `" Jon"` 和 `"Jon"` 在比较/JOIN 时可能被认为不同，所以 Silver 要先清理。


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Product Key Parsing

## 🔑 学习说明：解析 Product Key

源 `prd_key` 同时包含分类信息和商品编号。

这里把它拆成：
- `category_id`
- `product_number`

这样 Gold 层可以分别和 ERP 商品分类数据关联。


In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))


##Cost Cleanup

## 💰 学习说明：处理 Product Cost NULL

```text
prd_cost = NULL
→ 0
```

使用 `coalesce()` 给缺失成本提供默认值，避免后续计算/分析受 NULL 影响。


In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

##Product Line Normalization

## 🚲 学习说明：Product Line 标准化

把短代码统一成可读业务名称：

```text
M → Mountain
R → Road
S → Other Sales
T → Touring
```


In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

## Date Casting

## 📅 学习说明：商品日期转换

把源字段转换为 Spark `DateType`。

意义：
- 后续可以正确按日期排序/过滤
- 避免把日期当普通字符串处理


In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

## Renaming Columns

## 🏷️ 学习说明：统一字段名称

把源系统字段名统一成业务更容易理解的名字。

例如：

```text
cst_id        → customer_id
cst_key       → customer_number
prd_nm        → product_name
```

意义：Silver 不只是“清洗值”，也统一数据模型/字段契约。


In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Silver Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.silver.crm_products`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## Sanity checks of silver table

## ✅ 学习说明：验证 Silver Product

查询前 10 行确认：
- Product Key / Category 是否解析正确
- Cost / Product Line / Date 是否已经标准化
- Silver 表是否成功写入


In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10